## 1. Story and dataset

Clinical story (you can put this in the first markdown cell):

- We have patients with:
  - Tabular clinical variables: age, BMI, systolic BP.
  - Two imaging-derived biomarkers: LV ejection fraction (EF, regression) and LV mass index (regression).
  - Three binary risk flags from ECG/echo: AF present, LVH present, conduction abnormality (multi‑label, $C=3$).
- Outcomes:
  - Main task: 1‑year MACE risk (binary classification).
  - Secondary: future EF drop (regression).

We simulate this so you avoid loading real data.

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from trustcv import TrustCVValidator, UniversalCVRunner  # adjust import as in your repo

np.random.seed(42)
N = 800

# Tabular clinical features
age = np.random.normal(65, 10, N)
bmi = np.random.normal(27, 4, N)
sbp = np.random.normal(135, 15, N)

# Imaging biomarkers (continuous)
ef = np.random.normal(55, 8, N)        # %
lvmass = np.random.normal(90, 20, N)   # g/m2

# ECG/echo risk flags (multi-label C=3)
af_flag = (np.random.rand(N) < 0.25).astype(int)
lvh_flag = (np.random.rand(N) < 0.30).astype(int)
cond_flag = (np.random.rand(N) < 0.20).astype(int)
y_multi = np.stack([af_flag, lvh_flag, cond_flag], axis=1)

# Main binary outcome: 1y MACE risk (depends on age, EF, flags)
logit = (
    0.03*(age-60)           # age risk
    -0.05*(ef-55)           # higher EF protective
    +0.8*af_flag
    +0.5*lvh_flag
    +0.6*cond_flag
)
prob = 1/(1+np.exp(-logit))
y_main = (np.random.rand(N) < prob).astype(int)

# Secondary regression outcome: future EF drop
y_future_ef_drop = (np.maximum(0, 10 - 0.1*(ef-55)) 
                    + 2*af_flag 
                    + np.random.normal(0, 2, N))

# Assemble multi-input X representations
X_tab = np.stack([age, bmi, sbp], axis=1).astype("float32")
X_imaging = np.stack([ef, lvmass], axis=1).astype("float32")

# Options for passing to TrustCV
X_single = np.concatenate([X_tab, X_imaging], axis=1)          # simple
X_dict = {"clinical": X_tab, "imaging": X_imaging}             # dict input
X_list = [X_tab, X_imaging]                                    # list input


2026-01-31 12:12:27.834768: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-31 12:12:27.991399: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-31 12:12:28.194250: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-31 12:12:28.413525: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-31 12:12:28.414710: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-31 12:12:28.642467: I tensorflow/core/platform/cpu_feature_guard.cc:

In [2]:
print('\n--- X_imaging (Imaging Biomarkers) ---')
df_X_imaging = pd.DataFrame(X_imaging, columns=[f'imaging_biomarker_{i+1}' for i in range(X_imaging.shape[1])])
display(df_X_imaging.head())

print('\n--- X_single (Combined Features) ---')
df_X_single = pd.DataFrame(X_single, columns=[f'feature_{i+1}' for i in range(X_single.shape[1])])
display(df_X_single.head())

print('\n--- y_main (Main Binary Outcome) ---')
df_y_main = pd.DataFrame(y_main, columns=['y_main_outcome'])
display(df_y_main.head())

print('\n--- y_multi (Multi-label Outcome) ---')
df_y_multi = pd.DataFrame(y_multi, columns=['af_flag', 'lvh_flag', 'cond_flag'])
display(df_y_multi.head())

print('\n--- X_dict structure (First 2 rows of each array) ---')
for key, value in X_dict.items():
    print(f'Key: {key}')
    display(pd.DataFrame(value[:2], columns=[f'{key}_col_{i+1}' for i in range(value.shape[1])]))

print('\n--- X_list structure (First 2 rows of each array) ---')
for i, arr in enumerate(X_list):
    print(f'List Element {i+1}')
    display(pd.DataFrame(arr[:2], columns=[f'list_elem_{i+1}_col_{j+1}' for j in range(arr.shape[1])]))


--- X_imaging (Imaging Biomarkers) ---


,imaging_biomarker_1,imaging_biomarker_2
0,49.226101,93.703514
1,56.414566,98.508896
2,50.626560,94.445801
3,52.826759,115.577316
4,68.387619,70.953674



--- X_single (Combined Features) ---


,feature_1,feature_2,feature_3,feature_4,feature_5
0,69.967140,30.753136,132.256546,49.226101,93.703514
1,63.617355,24.935822,155.623154,56.414566,98.508896
2,71.476883,27.384483,125.310539,50.626560,94.445801
3,80.230301,25.150898,123.012123,52.826759,115.577316
4,62.658466,25.262014,127.758850,68.387619,70.953674



--- y_main (Main Binary Outcome) ---


...

,imaging_col_1,imaging_col_2
0,49.226101,93.703514
1,56.414566,98.508896



--- X_list structure (First 2 rows of each array) ---
List Element 1


,list_elem_1_col_1,list_elem_1_col_2,list_elem_1_col_3
0,69.967140,30.753136,132.256546
1,63.617355,24.935822,155.623154


List Element 2


,list_elem_2_col_1,list_elem_2_col_2
0,49.226101,93.703514
1,56.414566,98.508896


This lets you re‑use the same simulated cohort for all examples.

------

## 2. Simple regression (continuous y) via KerasSkWrap

Goal: show a plain regression head wrapped for TrustCV.

In [11]:
def build_regressor(input_dim):
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(32, activation="relu")(inputs)
    x = layers.Dense(16, activation="relu")(x)
    outputs = layers.Dense(1, activation="linear")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

reg_model = build_regressor(X_single.shape[1])

# If you have trustcv.keras.KerasSkWrap in your repo, use that wrapper.
from trustcv.frameworks.tensorflow_sklearn import KerasSkWrap  # adjust path if needed

reg_est = KerasSkWrap(
    build_fn=lambda: build_regressor(X_single.shape[1]),
    epochs=20,
    batch_size=32,
    verbose=0,
)



validator_reg = TrustCVValidator(method="kfold", n_splits=3, shuffle=True)


#res_reg = validator_reg.validate(reg_est, X_single, y_future_ef_drop)
res = validator_reg.validate(reg_est, X_single, y_future_ef_drop)
print(res.summary)

TypeError: TrustCVValidator.validate() takes 1 positional argument but 4 were given